# [Baseline] RandomForest — 학습

KBO 투구 하나가 **제구 성공** 투구일 확률을 예측하는 베이스라인입니다.

- **입력**: `test.csv` 의 47개 컬럼 (경기 상황, 투수·타자의 직전까지 누적 기록 등)
- **출력**: 제구 성공 확률 (0 이상 1 이하의 실수)
- **평가지표**: Brier Skill Score

`trackman_history.csv` 는 이 베이스라인에서 사용하지 않습니다. 2019~2024 과거 로그
179만 행이 그대로 남아 있으니 직접 활용해 보세요.

이 노트북은 모델을 **학습**하여 `./model/rf.pkl` 로 저장합니다. 저장한 모델은
추론용 `script.py` 와 함께 `baseline_submit.zip` 으로 묶어 제출합니다.

## 1. 라이브러리 불러오기

데이터 처리(pandas)와 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.
`joblib` 은 학습한 모델을 파일로 저장할 때 사용합니다.

In [22]:
import os
import time

import joblib
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin  # [신규] 커스텀 transformer용
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # [신규] HGB 비교 추가
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder  # [변경] OrdinalEncoder -> OneHotEncoder

DATA_DIR = "./data"

ID = "row_id"
TARGET = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]

## 2. 데이터 불러오기

`train.csv` 는 2019~2024 시즌이고 평가 데이터는 2025 시즌입니다.

사용할 피처 목록은 `test.csv` 가 정합니다. `train.csv` 에만 있는 컬럼을 학습에 넣으면
평가 시점에 그 컬럼이 없어 추론이 실패하기 때문입니다.

In [23]:
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])

print("train:", train.shape, "| 피처:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")

train: (1475092, 48) | 피처: 47 (범주형 3, 수치형 44)
시즌: 2019 ~ 2024
제구 성공률: 0.5238


## 3. 전처리 정의 — asof_* 스무딩 + 파생 피처 + 인코딩

범주형 3개(`top_bottom`, `game_type`, `base_state`)는 원-핫 인코딩하고, 수치형 컬럼의
결측값은 중앙값으로 채웁니다 (안전망 — 스무딩 이후엔 결측이 거의 남지 않습니다).

**`asof_*` 표본수 기반 Bayesian smoothing.** `asof_pitcher_n`, `asof_batter_n`,
`asof_pitcher_pitchmix_n`이 0인 행(신인/이적 초반, 학습 데이터 기준 각각 792/830/792행)은
관련 rate 컬럼이 결측입니다. 이걸 전체 median으로 채우면 신인에게 베테랑 수준 값을
부여하는 왜곡이 생기므로, 표본수를 신뢰도로 삼아 `(n*rate + k*prior) / (n+k)` 형태로
스무딩합니다 (`k=50`). cold-start 행은 `control_success` 평균이 오히려 전체보다 높았던
패턴을 모델이 학습하도록 `is_pitcher_cold_start` / `is_batter_cold_start` 플래그도
추가합니다. `asof_pitcher_prev1/3/5_game_*` 결측은 같은 지표의 스무딩된 누적값으로
대체합니다.

**[신규] 투수-타자 능력 차이 / 상황 압박 파생 피처.** RF는 얕은 트리(depth=10)라
"투수 능력 - 타자 능력" 같은 상호작용을 스스로 찾으려면 여러 번 split을 거쳐야 합니다.
`pitcher_batter_success_diff`, `pitcher_batter_middle_diff`(스무딩된 rate 차이),
카운트 압박(`count_pressure`, `is_full_count`, `is_two_strike_pressure`), 득점권 여부
(`is_scoring_position`), 좌우 매치업(`same_hand_matchup`)을 명시적으로 만들어 넣습니다.

**[변경] 범주형 인코딩: OrdinalEncoder → OneHotEncoder.** `base_state`, `game_type`,
`top_bottom`은 명목형(순서 없는 범주)인데 기존 OrdinalEncoder는 임의의 정수 순서를
부여했습니다. 카디널리티가 작아서(8/2/2, 총 12개 컬럼) OneHot으로 바꿔도 비용이 크지
않고, 트리가 특정 범주를 한 번의 split으로 분리할 수 있어 더 효율적일 수 있습니다.

스무딩 → 파생 피처 → 인코딩/대치 순서로 전부 파이프라인 안에 넣어서, 추론할 때도 학습
때와 동일한 로직이 그대로 따라가게 합니다.

In [24]:
# asof_* rate 컬럼의 cold-start 결측치를 표본수 기반으로 스무딩
class AsofRateSmoother(BaseEstimator, TransformerMixin):
    """표본수(n)가 작을수록 prior(학습 데이터 평균) 쪽으로, 클수록 실제 관측값
    쪽으로 끌어당기는 (n*rate + k*prior) / (n+k) 형태의 empirical Bayes smoothing.
    n=0(결측)이면 그대로 prior 값이 된다.
    """

    RATE_GROUPS = [
        ("asof_pitcher_n", ["asof_pitcher_success_rate", "asof_pitcher_reverse_rate",
                             "asof_pitcher_middle_rate", "asof_pitcher_ball_rate",
                             "asof_pitcher_strike_rate"]),
        ("asof_batter_n", ["asof_batter_success_rate", "asof_batter_middle_rate"]),
        ("asof_pitcher_pitchmix_n", ["asof_pitcher_fastball_rate",
                                      "asof_pitcher_breaking_rate",
                                      "asof_pitcher_offspeed_rate"]),
    ]
    # 직전 N경기 지표가 없으면(첫 등판 이후 두 번째 경기 전까지) 누적 지표로 대체
    PREV_GAME_FALLBACK = [
        ("asof_pitcher_prev1_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev3_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev5_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev1_game_middle_rate", "asof_pitcher_middle_rate"),
        ("asof_pitcher_prev3_game_middle_rate", "asof_pitcher_middle_rate"),
        ("asof_pitcher_prev5_game_middle_rate", "asof_pitcher_middle_rate"),
    ]
    SMOOTHING_K = 50  # prior를 표본 몇 개어치로 신뢰할지

    def fit(self, X, y=None):
        # prior는 fold의 학습 데이터에서만 계산 (검증/평가 데이터 누수 방지)
        self.priors_ = {
            c: X[c].mean()
            for _, rate_cols in self.RATE_GROUPS for c in rate_cols
        }
        return self

    def transform(self, X):
        X = X.copy()
        X["is_pitcher_cold_start"] = (X["asof_pitcher_n"] == 0).astype(int)
        X["is_batter_cold_start"] = (X["asof_batter_n"] == 0).astype(int)

        for n_col, rate_cols in self.RATE_GROUPS:
            n = X[n_col]
            for c in rate_cols:
                raw = X[c].fillna(0)
                X[c] = (n * raw + self.SMOOTHING_K * self.priors_[c]) / (n + self.SMOOTHING_K)

        for prev_col, fallback_col in self.PREV_GAME_FALLBACK:
            X[prev_col] = X[prev_col].fillna(X[fallback_col])

        return X


# [신규] asof_smooth 다음 단계 — 기존 컬럼을 조합한 파생 피처 추가
class DerivedFeatureBuilder(BaseEstimator, TransformerMixin):
    """투수-타자 능력 차이, 카운트/주자 압박, 좌우 매치업 등 도메인 지식 기반
    상호작용을 명시적으로 만든다. row별 계산이라 fit에서 저장할 상태가 없다.
    AsofRateSmoother 다음에 실행되어야 한다 (스무딩된 rate 값을 사용하므로).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["pitcher_batter_success_diff"] = (
            X["asof_pitcher_success_rate"] - X["asof_batter_success_rate"])
        X["pitcher_batter_middle_diff"] = (
            X["asof_pitcher_middle_rate"] - X["asof_batter_middle_rate"])

        X["count_pressure"] = X["balls_before"] - X["strikes_before"]
        X["is_full_count"] = ((X["balls_before"] == 3) & (X["strikes_before"] == 2)).astype(int)
        X["is_two_strike_pressure"] = (X["strikes_before"] == 2).astype(int)
        X["is_scoring_position"] = ((X["runner_on_2b"] == 1) | (X["runner_on_3b"] == 1)).astype(int)

        X["same_hand_matchup"] = (X["pitcher_hand"] == X["batter_hand"]).astype(int)
        return X


DERIVED_NUM_COLS = [
    "pitcher_batter_success_diff", "pitcher_batter_middle_diff",
    "count_pressure", "is_full_count", "is_two_strike_pressure",
    "is_scoring_position", "same_hand_matchup",
]

# [변경] 스무딩 플래그 2개 + 파생 피처 7개를 수치형 컬럼에 추가
NUM_COLS_EXT = NUM_COLS + ["is_pitcher_cold_start", "is_batter_cold_start"] + DERIVED_NUM_COLS

# [변경] OrdinalEncoder -> OneHotEncoder (명목형 범주라 순서 부여가 부적절했음)
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS_EXT),
])

## 4. 모델 학습과 검증 — Walk-forward CV + Recency Weighting

트리 깊이 10, 잎 노드 최소 샘플 200으로 얕게 제한합니다 (baseline 하이퍼파라미터, 이후 튜닝 대상).

기존에는 2019~2023으로 학습해 2024 한 시즌만 검증했습니다. `train.csv`의 연도별
제구 성공률이 2019년 0.565에서 2024년 0.486까지 꾸준히 낮아지는 추세라, 단일 연도
검증은 운(luck)에 따라 점수가 흔들릴 수 있습니다. 대신 시즌 기준
walk-forward(rolling-origin) 방식으로 과거 시즌들로 학습해 그다음 시즌을 맞히는 상황을
3번 반복합니다.

- Fold 1: train 2019~2021 → val 2022
- Fold 2: train 2019~2022 → val 2023
- Fold 3: train 2019~2023 → val 2024 (기존 baseline과 동일한 split)

**[신규] Recency weighting.** fold별 train/val Brier을 직접 비교해보니, val 점수가
fold마다 크게 흔들리는 원인이 overfitting이 아니라 **트렌드 추정 실패**였습니다:

| val 시즌 | 예측 평균 | 실제 평균 | 차이 | val 점수 |
|---|---|---|---|---|
| 2022 | 0.5305 | 0.5289 | +0.0016 | 2176.6 |
| 2023 | 0.5216 | 0.5000 | +0.0216 | 0.0 |
| 2024 | 0.5003 | 0.4861 | +0.0142 | 476.7 |

TRAIN 점수는 세 fold 모두 2173~2396으로 안정적인데, VAL만 요동칩니다. RF가 `season`을
정수 피처로만 다뤄서 훈련 구간 밖의 하락 추세를 못 뻗어나가고(extrapolate), 예측
평균이 실제보다 계속 높게 나오는 게 원인입니다(과대예측 폭이 클수록 점수가 더 크게
깎임). 이를 완화하기 위해 `RandomForestClassifier.fit()`에 `sample_weight`를 줘서
오래된 시즌의 영향력을 줄이고 최근 시즌 비중을 높입니다 (`season_sample_weight()`,
`half_life=2` — 2년마다 가중치 절반). 학습 데이터의 평균이 예측 대상 시즌에 더 가깝게
이동하길 기대합니다.

fold별 Brier Skill Score와 평균/표준편차를 함께 확인합니다.

**[신규] 모델 비교 — RandomForest vs HistGradientBoosting.** RF는 얕은 트리를 단순 평균 내는 구조라 Brier(확률 보정)에는 다소 불리할 수 있습니다. 반면 `HistGradientBoostingClassifier`는 log-loss를 직접 최적화하며 순차적으로 잔차를 학습하기 때문에 확률 보정이 더 좋고, 얕은 트리만으로도 상호작용을 잘 잡아내는 경향이 있습니다. scikit-learn 내장이라 `requirements.txt`에 새 의존성을 추가하지 않아도 됩니다. `make_model(model_type=...)`으로 `"rf"`/`"hgb"`를 골라서 동일한 전처리·walk-forward CV·recency weighting 위에서 그대로 비교할 수 있습니다.

In [ ]:
# ── [변경] 단일 split(2019~2023 학습 / 2024 검증) → walk-forward CV로 교체 ──

# [신규] model_type으로 RF <-> HistGradientBoosting을 바로 교체해서 비교
MODEL_TYPE = "hgb"  # "rf"(기존 baseline) 또는 "hgb"(채택, 실제 제출 810점)


def make_model(model_type=MODEL_TYPE):
    """fold마다 새로 학습할 파이프라인(학습 전 상태)을 만든다.

    전처리(asof_smooth/derived/pre)는 두 모델이 동일하게 공유하므로
    model_type만 바꾸면 같은 walk-forward CV로 RF와 HGB를 바로 비교할 수 있다.
    """
    if model_type == "rf":
        clf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=200,
            n_jobs=-1,
            random_state=42,
        )
    elif model_type == "hgb":
        # [주의] `## 4-1.` 그리드서치로 찾은 lr=0.03/max_leaf_nodes=63/min_samples_leaf=400
        # 조합은 CV에서는 더 높았지만(981.79) 실제 제출은 767점으로 810점보다 하락해서
        # 되돌림 — CV 점수 히스토리 "그리드서치 결과가 실제 평가와 반대로 간 사례" 참고.
        clf = HistGradientBoostingClassifier(
            max_iter=100,           # RF의 n_estimators=100에 대응
            learning_rate=0.1,
            min_samples_leaf=200,   # RF와 동일한 잎 노드 최소 샘플로 정규화 강도를 맞춤
            early_stopping=False,   # walk-forward CV 자체가 검증 역할을 하므로 내부 조기종료는 끔
            random_state=42,
        )
    else:
        raise ValueError(f"알 수 없는 model_type: {model_type!r}")

    return Pipeline([
        ("asof_smooth", AsofRateSmoother()),
        ("derived", DerivedFeatureBuilder()),
        ("pre", preprocessor),
        ("clf", clf),
    ])


def brier_skill_score(y_true, y_pred):
    """기존 5번 섹션에 있던 계산식을 함수로 분리 — fold마다 재사용."""
    r = y_true.mean()
    brier = ((y_pred - y_true) ** 2).mean()
    baseline_brier = r * (1 - r)
    return max(0, 100000 * (1 - brier / baseline_brier))


# [신규] 최근 시즌일수록 학습 가중치를 높인다 (half_life년마다 가중치 절반).
# 기준점은 학습 데이터의 마지막 시즌 = 예측 대상 바로 이전 해.
def season_sample_weight(seasons, half_life=2):
    max_season = seasons.max()
    return 0.5 ** ((max_season - seasons) / half_life)


# walk-forward fold 정의: (학습에 쓸 시즌들, 검증할 다음 시즌)
FOLDS = [
    (range(2019, 2022), 2022),  # train 2019~2021 -> val 2022
    (range(2019, 2023), 2023),  # train 2019~2022 -> val 2023
    (range(2019, 2024), 2024),  # train 2019~2023 -> val 2024 (기존 baseline과 동일)
]

# fold를 순회하며 매번 새 모델을 학습하고 점수를 기록
scores = []
for train_seasons, val_season in FOLDS:
    is_train = train["season"].isin(train_seasons)
    is_val = train["season"] == val_season
    X_train, y_train = train.loc[is_train, FEATURES], train.loc[is_train, TARGET]
    X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET]

    fold_model = make_model()
    t = time.time()
    train_weight = season_sample_weight(train.loc[is_train, "season"])  # [신규]
    fold_model.fit(X_train, y_train, clf__sample_weight=train_weight)  # [변경]
    val_pred = fold_model.predict_proba(X_val)[:, 1]
    score = brier_skill_score(y_val, val_pred)
    scores.append(score)

    print(f"[val={val_season}] train={len(X_train)} val={len(X_val)} "
          f"score={score:.2f} ({time.time() - t:.1f}s)")

# fold 평균/표준편차로 점수 안정성 확인 (한 시즌만 봤을 때보다 신뢰도 높음)
mean_score = sum(scores) / len(scores)
std_score = (sum((s - mean_score) ** 2 for s in scores) / len(scores)) ** 0.5
print(f"\nCV Score: {mean_score:.2f} (mean) ± {std_score:.2f} (std) | "
      f"fold별: {[round(s, 2) for s in scores]} | model_type={MODEL_TYPE}")

# 다음 섹션(5. 전체 재학습)에서 쓸 파이프라인 — 아직 학습 전 상태로 정의만 해둠
# (fold_model들과는 별개 객체. 다음 셀에서 전체 데이터로 fit 됨)
model = make_model()

## 5. 전체 데이터로 재학습 & 모델 저장

검증으로 성능을 확인했으니 이제 전체 학습 데이터로 다시 학습합니다.

**[변경]** 여기서도 CV와 동일하게 `season_sample_weight()`로 최근 시즌(2024) 비중을
높여서 학습합니다. 실제 평가 대상인 2025년이 2024년 바로 다음 해라, 2024와 가장 가까운
가중치 분포로 학습하는 게 CV 때와 같은 논리입니다.

학습한 파이프라인을 `./model/rf.pkl` 로 저장합니다. 이 파일을 추론용 `script.py`,
`requirements.txt` 와 함께 `baseline_submit.zip` 으로 묶으면 제출 준비가 끝납니다.

In [ ]:
t = time.time()
final_weight = season_sample_weight(train["season"])  # [신규] CV와 동일한 recency weighting
model.fit(train[FEATURES], train[TARGET], clf__sample_weight=final_weight)  # [변경]
print(f"재학습 완료 :: {time.time() - t:.1f}s")

# [신규] HistGradientBoostingClassifier는 fit 상태에 numpy Generator(PCG64) 객체를
# _feature_subsample_rng로 들고 있는데(predict에는 안 쓰임, fit 전용), 이게 pickle에
# 그대로 들어가면 채점 서버 numpy 버전이 학습 환경과 다를 때
# "not a known BitGenerator module" 에러로 unpickle이 깨진다. predict에 영향 없으니
# 저장 전에 제거한다.
clf = model.named_steps["clf"]
if hasattr(clf, "_feature_subsample_rng"):
    del clf._feature_subsample_rng

os.makedirs("./model", exist_ok=True)
joblib.dump(model, "./model/rf.pkl", compress=3)
print("저장 완료: ./model/rf.pkl")